# Relationship Configuration
Inspect the centralized dimension registry, custom relationships, table overrides, and approved relationship context before discovery.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent.config import TablePair, load_app_config, load_business_context
from dq_agent.context_store import read_context
from dq_agent.context_utils import configure_workflow_logging, workflow_paths, logged_step
from dq_agent.relationships import feature_flag_for_pair, load_relationship_configuration

In [ ]:
RUN_ID = 'relationship_config_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
config = load_app_config(ROOT)
paths = workflow_paths(config, RUN_ID, 'relationships')
logger = configure_workflow_logging(paths['log'], config.project.log_level)
business_context = load_business_context(config)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_RELATIONSHIP_CONFIGURATION', file=config.project.relationships):
    relationship_config = load_relationship_configuration(config)
    print(relationship_config.settings.model_dump())

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_DIMENSION_REGISTRY'):
    dimension_rows = []
    for name, dimension in relationship_config.dimensions.items():
        for key in dimension.keys:
            dimension_rows.append({
                'dimension': name, 'table': dimension.table,
                'parent_columns': key.parent_columns,
                'child_column_sets': key.child_column_sets,
                'parent_filters': dimension.parent_filters, 'enabled': dimension.enabled,
            })
    dimensions = pd.DataFrame(dimension_rows)
    logger.info('LOAD_DIMENSION_REGISTRY dimensions=%s key_definitions=%s', len(relationship_config.dimensions), len(dimensions))
display(dimensions)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'LOAD_CUSTOM_RELATIONSHIPS'):
    custom_relationships = pd.DataFrame([item.model_dump() for item in relationship_config.custom_relationships])
    logger.info('LOAD_CUSTOM_RELATIONSHIPS records=%s', len(custom_relationships))
display(custom_relationships)

In [ ]:
sample_pairs = [
    TablePair(pair_id='sample_fact_sales', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='fact_sales'),
    TablePair(pair_id='sample_dim_market', mode='bigquery_only', target_project='your-gcp-project', target_dataset='analytics', target_table='dim_market'),
]
with logged_step(logger, paths['checkpoint'], 'APPLY_FEATURE_FLAGS'):
    flag_rows = []
    for pair in sample_pairs:
        table_context = business_context.get('tables', {}).get(pair.pair_id, {})
        enabled, source = feature_flag_for_pair(pair, relationship_config, table_context)
        flag_rows.append({'pair_id': pair.pair_id, 'auto_inference_enabled': enabled, 'source': source})
    flags = pd.DataFrame(flag_rows)
display(flags)

In [ ]:
with logged_step(logger, paths['checkpoint'], 'RETRIEVE_APPROVED_RELATIONSHIPS'):
    trusted = read_context(config, logger=logger)
    approved_relationships = trusted[trusted['context_type'] == 'relationship'] if not trusted.empty else pd.DataFrame()
    logger.info('RETRIEVE_APPROVED_RELATIONSHIPS records=%s', len(approved_relationships))
display(approved_relationships)

Custom relationships remain active even when automatic inference is disabled. Per-table business context overrides `relationships.yaml`, which overrides the global flag.